<a href="https://colab.research.google.com/github/Sumit-Todwal/Assignments_CEI_2026/blob/main/Week7_Sumit_Todwal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 — Retrieval-Augmented Generation (RAG) System
### Document Question Answering over Custom PDFs —

**Author:** Sumit Todwal

RAG pipeline from scratch:
1. Document Ingestion (PDF → text)
2. Text Chunking
3. Embedding Creation (Sentence-Transformers)
4. Vector Database (FAISS)
5. Query Embedding + Retrieval
6. Answer Generation (FLAN-T5, free local model)


In [28]:
!pip install pypdf sentence-transformers faiss-cpu transformers torch --quiet

In [29]:
import os
import numpy as np
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Step 1: Document Ingestion

In [30]:
from google.colab import files

uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {PDF_PATH}")

Saving Sumit_Todwal_Resume_July_2026.pdf to Sumit_Todwal_Resume_July_2026 (1).pdf
Uploaded: Sumit_Todwal_Resume_July_2026 (1).pdf


In [31]:
def load_pdf(file_path):
    """Extract raw text from a PDF file, page by page."""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

raw_text = load_pdf(PDF_PATH)
print(f"Extracted {len(raw_text)} characters from the document.")
print(raw_text[:500])

Extracted 3111 characters from the document.
Sumit Todwal
+91-7877526566|todwalsumit@gmail.com |linkedin.com/in/sumit-todwal-810905350 |github.com/Sumit-Todwal |Jaipur, Rajasthan
Summary
Final-year CSE student (CGPA: 9.04/10) with hands-on experience in Python, SQL, ETL pipelines, and full-stack development.
Delivered production-grade distributed systems (500+ tasks/min) and IoT pipelines (10,000+ data points/hour). Solved 250+ DSA
problems on LeetCode (top 37%); selected for McKinsey Forward Program 2025.
Education
Swami Keshvanand Instit


## Step 2: Text Chunking
Large documents need to be split into smaller overlapping chunks. This improves retrieval
accuracy since embeddings work better on focused, smaller pieces of text than on entire
documents. Overlap prevents losing context at chunk boundaries.

In [32]:
def chunk_text(text, chunk_size=150, overlap=30):
    """
    Split text into overlapping word-based chunks.
    chunk_size : number of words per chunk
    overlap    : number of words shared between consecutive chunks
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text, chunk_size=150, overlap=30)
print(f"Document split into {len(chunks)} chunks.")
print("\nSample chunk:\n", chunks[0])

Document split into 4 chunks.

Sample chunk:
 Sumit Todwal +91-7877526566|todwalsumit@gmail.com |linkedin.com/in/sumit-todwal-810905350 |github.com/Sumit-Todwal |Jaipur, Rajasthan Summary Final-year CSE student (CGPA: 9.04/10) with hands-on experience in Python, SQL, ETL pipelines, and full-stack development. Delivered production-grade distributed systems (500+ tasks/min) and IoT pipelines (10,000+ data points/hour). Solved 250+ DSA problems on LeetCode (top 37%); selected for McKinsey Forward Program 2025. Education Swami Keshvanand Institute of T echnology (SKIT)Aug 2023 – Present B.Tech in Computer Science and Engineering (CGPA: 9.04/10) Jaipur, Rajasthan Emmanuel Mission Sr. Sec. School Class XII: 76.4%|Class X: 87.5% (RBSE Board) Niwai, Rajasthan Technical Skills Languages:Python, C++, SQL, JavaScript, HTML, CSS F rameworks & T ools:FastAPI, Flask, Pandas, NumPy, Docker, GitHub Actions, Git, Power BI, Raspberry Pi Databases & Cloud:SQLite, Databricks Core Concepts:DSA, ETL Pipelin

## Step 3: Embedding Creation
Each chunk is converted into a dense vector using a pretrained sentence embedding model
(`all-MiniLM-L6-v2` — small, fast, free). On Colab's GPU this is near-instant even for
large documents.

In [33]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print("Embedding matrix shape:", chunk_embeddings.shape)  # (num_chunks, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (4, 384)


## Step 4: Vector Database (FAISS)
FAISS lets us do fast similarity search over the chunk embeddings. We use a simple
`IndexFlatL2` (exact search via Euclidean distance) — perfect for small documents.
For larger corpora, an approximate index (e.g. `IndexIVFFlat`) would be faster.

In [34]:
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}.")

FAISS index built with 4 vectors of dimension 384.


## Step 5 & 6: Query Embedding + Context Retrieval
Convert the user's question into the same embedding space, then find the `k` most similar
chunks in the FAISS index. These chunks become the "context" for the generation step.

In [35]:
def retrieve_context(query, k=3):
    """Return the top-k most relevant chunks for a given query."""
    query_embedding = embedding_model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    return retrieved_chunks, distances[0]

# quick test
test_chunks, test_scores = retrieve_context("What are the technical skills mentioned?", k=3)
for i, c in enumerate(test_chunks):
    print(f"--- Retrieved chunk {i+1} (distance={test_scores[i]:.2f}) ---")
    print(c[:200], "...\n")

--- Retrieved chunk 1 (distance=1.33) ---
Sumit Todwal +91-7877526566|todwalsumit@gmail.com |linkedin.com/in/sumit-todwal-810905350 |github.com/Sumit-Todwal |Jaipur, Rajasthan Summary Final-year CSE student (CGPA: 9.04/10) with hands-on exper ...

--- Retrieved chunk 2 (distance=1.42) ---
Distributed Systems Work Experience Coplur Jun 2025 – Aug 2025 Data Analytics Intern Jaipur, Rajasthan •Engineeredautomated ETL pipelines in Python (Pandas, NumPy) and SQL, streamlining manual data pr ...

--- Retrieved chunk 3 (distance=1.44) ---
and anomaly detection on large-scale datasets. •LeetCode:Contest rating 1525 (top 37.32%); solved 250+ DSA problems across competitive programming platforms. •Advanced C++ Certification (95%):Proficie ...



## Step 7: Answer Generation
We feed the retrieved chunks + the question into a free, local language model
(`google/flan-t5-base`) using a text2text-generation pipeline, running on GPU if available.
The model is instructed to answer **only** using the provided context — this is what keeps
RAG answers grounded and reduces hallucination.

In [36]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

def generate_answer(query, k=3, max_new_tokens=150):
    retrieved_chunks, _ = retrieve_context(query, k=k)
    context = "\n\n".join(retrieved_chunks)

    prompt = (
        f"Answer the question using only the context below. "
        f"If the answer is not in the context, say 'I don't know based on the document.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        f"Answer:"
    )

    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer, retrieved_chunks

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## Step 8: Try It Out
Ask questions about your own document. Try a few different kinds of questions —
factual lookups work best (RAG is not great at broad summarization with small models
like flan-t5-base, but works well for "what/where/when" style questions).

In [37]:
questions = [
    "What is this document about?",
    "What programming languages or skills are mentioned?",
    "What projects are described?"
]

for q in questions:
    answer, sources = generate_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

Q: What is this document about?
A: Science/Tech
------------------------------------------------------------
Q: What programming languages or skills are mentioned?
A: Python, C++, SQL, JavaScript, HTML, CSS
------------------------------------------------------------
Q: What projects are described?
A: Distributed T ask Queue System|Python, FastAPI, SQLite, Docker|Live Demo |GitHub
------------------------------------------------------------


## Step 9: Interactive Q&A Loop (optional)
Run this cell to ask your own questions in a loop. Type `exit` to stop.

In [38]:
while True:
    user_q = input("Ask a question (or 'exit'): ")
    if user_q.strip().lower() == "exit":
        break
    ans, srcs = generate_answer(user_q)
    print("Answer:", ans)
    print()

Ask a question (or 'exit'): How many internships does user have?
Answer: I don't know based on the document.

Ask a question (or 'exit'): projects of user?
Answer: Distributed T ask Queue System|Python, FastAPI, SQLite, Docker|Live Demo |GitHub

Ask a question (or 'exit'): skills of user?
Answer: Python, C++, SQL, JavaScript, HTML, CSS

Ask a question (or 'exit'): Certifications achieved by user
Answer: Advanced C++ Certification (95%)

Ask a question (or 'exit'): Overall Score for this Resume.
Answer: 9.04/10

Ask a question (or 'exit'): Classes present in this resume.
Answer: B.Tech in Computer Science and Engineering

Ask a question (or 'exit'): School of user?
Answer: SKIT

Ask a question (or 'exit'): How user related to Emmanuel?
Answer: I don't know based on the document.

Ask a question (or 'exit'): How much accuracy does user have?
Answer: 99%+ task completion reliability

Ask a question (or 'exit'): exit


## Notes / Possible Improvements
- **Better chunking:** try sentence-aware or paragraph-aware chunking instead of fixed word counts.
- **Better embeddings:** try `all-mpnet-base-v2` (more accurate, slower) instead of MiniLM.
- **Hybrid search:** combine keyword search (BM25) with vector search for better retrieval.
- **Re-ranking:** use a cross-encoder to re-rank top-k retrieved chunks before generation.
- **Bigger LLM:** swap `flan-t5-base` for `flan-t5-large` (Colab GPU handles it easily) for better answer quality.
- **Multi-document support:** loop `load_pdf` + `chunk_text` over multiple uploaded files, keep a `source` field per chunk so you know which document an answer came from.
- **Persisting across sessions:** Colab runtime resets, so save `chunk_embeddings`/`chunks` to Google Drive if you want to reuse them without re-embedding every time.